# 01 · Module 1: Strategic capital allocation

**Question.** Which initiatives NovaFin should fund, and how much capital that takes.

The answer is driven more by the NPV assumptions than by the success model: show both and say which matters more.

**Data.** `strategic_initiatives.csv`: 180 initiatives in ₹ lakh (pipeline about ₹283 cr), Historical_Success 77%. Split: repeated stratified 5-fold CV, 20% hold-out.

**Professor's anchor.** S5: NPV, IRR, WACC, discount-rate sensitivity, Monte Carlo (ElectraCharge).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("01")

In [ ]:
ini = load("initiatives")
ini.head()

> Evaluation standard: a naive baseline and a transparent model before any tree; PR-AUC against the base rate, Brier score and a calibration check for classifiers; the threshold chosen on validation data and reported on test data; SHAP or coefficients for the model that drives the decision; a stability check by period, segment or stock.

## 1. EDA (Task 1)

Distributions of investment and ROI; success rate by Risk_Rating and Business_Unit; Business_Growth against Expected_ROI; success rate by investment size band.

## 2. Success model (Task 2)

Logistic regression (scaled, one-hot Business_Unit and Risk_Rating) against random forest and gradient boosting, with `splits.repeated_cv()`. With 180 rows expect the logistic model to match the trees; say so. Calibrate the chosen model: its probability feeds expected NPV.

## 3. Explainability

`explain.coefficient_table` and SHAP. Check the signs: higher ROI and growth up, higher competition down (S12: a sign against theory is hard to defend).

## 4. NPV and IRR (Task 3)

CF_t = Revenue_t − Operating_Cost. Rate = Interest_Rate + premium by Risk_Rating (`ASSUMPTIONS["RISK_PREMIUM_PTS"]`). Terminal value: year-3 cash flow growing at 3%, with 0% and 5% as sensitivities (`finance.npv`, `finance.irr`).

## 5. Expected NPV

eNPV = P(success) × NPV as the brief defines it, plus the variant in which failure loses part of the investment (`finance.expected_npv`).

## 6. Monte Carlo

Draw revenues around their values (±15%) and the discount rate across its range; report P(NPV < 0) per initiative. Seed 42.

## 7. Decision

**Fund:** eNPV in the top tier, P(NPV < 0) below 20%, Risk_Rating not High unless eNPV is exceptional. **Review:** positive eNPV but P(NPV < 0) of 20–40%, or High risk, or the decision flips under the terminal-value sensitivity. **Reject:** eNPV ≤ 0 or P(NPV < 0) above 40%.

## Handoff to Module 9

Capital needed for the Fund list (at most ₹283 cr), portfolio eNPV per rupee, and the Review list.

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("01", lines)

## Traps to check before handing over

Without a terminal value the 3-year horizon makes nearly every NPV negative. Expected_ROI may already be a forward estimate correlated with the cash flows: keep it, but flag it.